In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations 
from sklearn.cluster import KMeans,AgglomerativeClustering
from sklearn.preprocessing import  StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score,mutual_info_score,homogeneity_score,homogeneity_completeness_v_measure
import numpy as np
from yellowbrick.cluster import KElbowVisualizer

In [ ]:
import pandas as pd

# Load the CSV with semicolon field separator, comma as decimal point,
# and using Windows-1250 encoding (common for Central European files)
df = pd.read_csv(
    'stock_exchanges_data.csv',
    sep=';',           # fields are separated by semicolons
    decimal=',',       # decimal values use comma
    encoding='cp1250'  # adjust if your file uses a different encoding
)

# Quick sanity check: show the first 5 rows
print(df.head())


In [ ]:
import pandas as pd

# 1. Load the CSV with semicolon field separator and Windows-1250 encoding
df = pd.read_csv(
    'stock_exchanges_data.csv',
    sep=';',            
    encoding='cp1250'   
)

# 2. Identify all columns that should be numeric but currently are strings
#    (you can also explicitly list them if you know which ones they are)
numeric_cols = df.select_dtypes(include='object').columns

# 3. Replace comma decimal separators with dots, then convert to float
for col in numeric_cols:
    # strip spaces, replace ','→'.', then coerce to numeric
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(',', '.', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )

# 4. Quick check: show dtypes and first few rows
print("Column data types after conversion:")
print(df.dtypes, "\n")
print("Preview of cleaned data:")
print(df.head())


In [ ]:
import numpy as np
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler

# 1. Select the variables to include in the clustering model
data_to_model = df[
    [
        'Capitalization',
        'Capitalization/GDP',
        'Value traded (EOB Total)',
        'Value traded (Total)/GDP',
        'Share turnover velocity',
        'Capitalization/Number of listed companies (Total)',
        'Number of listed companies (Total)',
        'Number of listed companies (Foreign)/Number of listed companies (Total)',
        'Number of listed companies (Domestic)/Population_2022',
        'Number of listed companies (Domestic)',
        'Number of listed companies (Foreign)',
        'Number of new listings through IPO (Total)',
        'Number of trades (EOB)',
        'GDP_2022',
        'Population_2022'
    ]
]

# 2. Standardize features to mean=0 and variance=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data_to_model)
scaled_df = pd.DataFrame(
    X_scaled,
    columns=data_to_model.columns,
    index=data_to_model.index
)

# 3. Compute squared Mahalanobis distance for each observation
#    After standardization, the data have mean zero
cov_matrix     = np.cov(scaled_df.values, rowvar=False)
inv_cov_matrix = np.linalg.inv(cov_matrix)
md2 = np.einsum(
    'ij,jk,ik->i',
    scaled_df.values,
    inv_cov_matrix,
    scaled_df.values
)

# 4. Define outlier cutoff from χ² distribution (e.g. 5% significance)
alpha     = 0.001
p         = scaled_df.shape[1]             # number of variables
cutoff    = chi2.ppf(1 - alpha, df=p)

# 5. Filter out observations exceeding the cutoff
mask          = md2 <= cutoff
cleaned_data  = data_to_model.loc[mask].copy()

# 6. (Optional) Report how many records were removed
n_removed  = (~mask).sum()
n_retained = mask.sum()
print(f"Removed {n_removed} outliers; retained {n_retained} records.")


In [ ]:
import itertools
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# ——————————————————————————————————————————————————————————————————————
# ASSUMPTIONS:
# • `cleaned_data` is your DataFrame after outlier removal, containing only the
#   variables you want to test.
# • You have already imported pandas and numpy earlier in your script.
# ——————————————————————————————————————————————————————————————————————

# 1. Re‐standardize the cleaned data for clustering
scaler = StandardScaler()
X_clean_scaled = scaler.fit_transform(cleaned_data)
scaled_cleaned_df = pd.DataFrame(
    X_clean_scaled,
    columns=cleaned_data.columns,
    index=cleaned_data.index
)

# 2. Set up a list to hold (variable_subset, best_silhouette, optimal_k)
results = []

# 3. Exhaustive search: for every combination of variables (size ≥2),
#    compute silhouette scores for k = 2…10 and record the best.
vars_list = cleaned_data.columns.tolist()

for r in range(2, len(vars_list) + 1):
    for subset in itertools.combinations(vars_list, r):
        X_subset = scaled_cleaned_df[list(subset)].values
        
        best_score = -1.0
        best_k     = None
        
        # test cluster counts from 2 up to 10
        for k in range(2, 11):
            kmeans = KMeans(n_clusters=k, random_state=42).fit(X_subset)
            labels = kmeans.labels_
            score  = silhouette_score(X_subset, labels)
            
            if score > best_score:
                best_score = score
                best_k     = k
        
        results.append({
            'variables': subset,
            'silhouette_score': best_score,
            'optimal_k': best_k
        })

# 4. Identify which variable subset achieved the highest silhouette
best_result = max(results, key=lambda x: x['silhouette_score'])

print(f"Best silhouette: {best_result['silhouette_score']:.3f}")
print(f"Optimal number of clusters: {best_result['optimal_k']}")
print("Variables in best subset:")
for var in best_result['variables']:
    print(f" - {var}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples

# 1. Specify the variables you determined were best
best_vars = [
            'Capitalization', 
            'Capitalization/GDP',
            'Value traded (EOB Total)', 
            'Value traded (Total)/GDP',
            'Share turnover velocity',
            'Capitalization/Number of listed companies (Total)',
            'Number of trades (EOB)'
]

# 2. Extract the already-standardized feature matrix
X = data_to_model[best_vars].values

print("Variables used for clustering:", best_vars)

# 3. Loop over k = 2…10 to compute:
#    - silhouette for KMeans and Ward
#    - inertia (distortion) for KMeans
clusters_range = range(2, 11)
kmean_sil, ward_sil, inertia = [], [], []

for k in clusters_range:
    # K-Means
    km = KMeans(n_clusters=k, init='random', n_init=100, max_iter=100, random_state=42)
    labels_km = km.fit_predict(X)
    kmean_sil.append(silhouette_score(X, labels_km))
    inertia.append(km.inertia_)
    
    # Ward linkage
    ward = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels_ward = ward.fit_predict(X)
    ward_sil.append(silhouette_score(X, labels_ward))
    
    print(f"Clusters={k}  |  Silhouette KMeans={kmean_sil[-1]:.3f}  |  Silhouette Ward={ward_sil[-1]:.3f}")

# 4. Build a DataFrame of results
sil_df = pd.DataFrame({
    'clusters': list(clusters_range),
    'kmeans_sil': kmean_sil,
    'ward_sil': ward_sil,
    'inertia': inertia
})
display(sil_df)

# 5. Plot silhouette coefficients (KMeans)
plt.figure(figsize=(8, 4))
plt.plot(sil_df['clusters'], sil_df['kmeans_sil'],
         marker='o', linestyle='-', color='black', label='K-Means')
# plt.plot(sil_df['clusters'], sil_df['ward_sil'],
#          marker='s', linestyle='--', color='black', label='Ward')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette coefficient')
plt.title('Silhouette Analysis')
plt.legend()
plt.show()

# 6. Plot elbow (inertia)
plt.figure(figsize=(8, 4))
plt.plot(sil_df['clusters'], sil_df['inertia'],
         marker='o', linestyle='-', color='black')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia (distortion)')
plt.title('Elbow Method')
plt.show()

# 7. Choose the k with highest KMeans silhouette
opt_k = int(sil_df.loc[sil_df['kmeans_sil'].idxmax(), 'clusters'])
print(f"Optimal number of clusters (by silhouette): {opt_k}")

# 8. Fit final KMeans, compute per-sample silhouette, and tabulate
final_km = KMeans(n_clusters=opt_k, init='random', n_init=100, max_iter=100, random_state=42)
final_labels = final_km.fit_predict(X)
final_silh   = silhouette_samples(X, final_labels)

results = pd.DataFrame({
    'ExchangeName': data_to_model.index,  # or use your original df['ExchangeName']
    'silhouette': final_silh,
    'cluster': final_labels
})
results.head()


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
import shap

data_to_rf = data_to_model.copy()
data_to_rf['cluster'] = results['cluster'].values

# Separate the dataset into input features (X_train) and target variable (y_train)
# Features: all columns except the last one
# Target: 'cluster' column indicating the cluster assignment
X_train = data_to_rf.iloc[:, :-1]
y_train = data_to_rf['cluster'].values

# Initialize a Random Forest Classifier
# The model will later be tuned using cross-validation
rf = RandomForestClassifier(
    random_state=42  # Ensuring reproducibility of results
    # Additional parameters can be added here if necessary
)

# Define the hyperparameter grid to optimize the Random Forest
# The grid explores variations in:
# - number of trees (n_estimators),
# - maximum depth of trees (max_depth),
# - minimum number of samples required to split an internal node (min_samples_split),
# - minimum number of samples required to be at a leaf node (min_samples_leaf)
param_grid = {
    'n_estimators': [PLACEHOLDER_FOR_N_ESTIMATORS],        # e.g., [50, 100, 200]
    'max_depth': [PLACEHOLDER_FOR_MAX_DEPTH_VALUES],       # e.g., [10, 12, 20, 30]
    'min_samples_split': [PLACEHOLDER_FOR_MIN_SPLIT],      # e.g., [2, 5, 10]
    'min_samples_leaf': [PLACEHOLDER_FOR_MIN_LEAF]         # e.g., [1, 2, 4]
}

# Perform hyperparameter tuning using GridSearchCV
# Cross-validation (cv) ensures the model is evaluated on different splits of the data
cv_rf = GridSearchCV(
    rf,
    param_grid,
    cv=PLACEHOLDER_FOR_CV_FOLDS  # e.g., cv=2
)
cv_rf.fit(X_train, y_train)  # Training the model with all parameter combinations

# Retrieve the best model identified during the grid search
best_rf = cv_rf.best_estimator_

# Output the best hyperparameters found
print(cv_rf.best_params_)

# Initialize the SHAP (SHapley Additive exPlanations) explainer
# TreeExplainer is optimized for tree-based models such as Random Forest
explainer = shap.TreeExplainer(best_rf)

# Compute SHAP values for all samples in the training set
# These values explain the contribution of each feature to the prediction
shap_values = explainer.shap_values(X_train)

# Generate a summary plot of SHAP values
# The bar plot type aggregates mean absolute SHAP values per feature
# and ranks features by their overall impact on the model
shap.summary_plot(shap_values, X_train, plot_type="bar")


In [ ]:
import matplotlib.pyplot as plt
import shap

# Generate Figures 3 to 7:
# For each cluster (0 to 4), we create two SHAP summary plots:
# - Left: standard dot plot showing individual feature impacts
# - Right: bar plot showing mean absolute SHAP values
# Each figure is saved separately as a high-quality PNG file.

# Iterate over cluster indices
for cluster_idx in range(5):
    # Create a figure with two subplots: one for the dot plot, one for the bar plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), gridspec_kw={'width_ratios': [1, 1]}, sharey=True)

    # Left plot: SHAP dot plot
    plt.sca(axes[0])  # Focus on the first subplot
    shap.summary_plot(shap_values[:, :, cluster_idx], X_train, show=False)
    axes[0].set_title('Impact on Model Output')
    axes[0].set_xlabel("SHAP value")

    # Remove the 'Feature value' text automatically added by SHAP
    texts = plt.gcf().texts
    for text in texts:
        if 'Feature value' in text.get_text():
            text.remove()

    # Right plot: SHAP bar plot
    plt.sca(axes[1])  # Focus on the second subplot
    shap.summary_plot(shap_values[:, :, cluster_idx], X_train, plot_type="bar", show=False)
    axes[1].set_title('Average Impact on Model Output Magnitude')
    axes[1].set_xlabel("mean(|SHAP value|)")

    # Add a shared y-axis label ('Feature') for both subplots
    fig.text(0.04, 0.5, 'Feature', va='center', rotation='vertical', fontsize=12)

    # Save each figure separately, e.g., Fig3.png, Fig4.png, etc.
    fig.savefig(f"Fig{3 + cluster_idx}.png", dpi=300, bbox_inches='tight')

    # Display the figure
    plt.show()


In [ ]:
import pandas as pd

# Load the original dataset containing financial indicators for global stock exchanges.
# This file provides the basis for clustering, classification, and SHAP analysis.
data = pd.read_csv('stock_exchanges_data.csv', index_col=0)

# This prepares a complete dataset ready for classification and interpretability analysis
data['cluster'] = results['cluster'].values


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Generate Figure 8:
# This map visualizes the global distribution of clusters assigned to stock exchanges.
# Open-source geographic data from the 'Natural Earth' dataset (via GeoPandas) is used
# to ensure reproducibility and transparency.

# Prepare the cluster assignment data
df = data[['Country', 'cluster']]

# Add missing country information manually if necessary
# (e.g., Greenland not covered in the original dataset)
additional_data = pd.DataFrame({'Country': ['Greenland'], 'cluster': [2]})
df = pd.concat([df, additional_data], ignore_index=True)

# Some country names need to be standardized to match the map dataset
country_renames = {
    'Islamic Republic of Iran': 'Iran',
    'Korea': 'South Korea',
    'United States': 'United States of America',
    'Czech Republic': 'Czechia',
    'Taiwan Province of China': 'Taiwan',
    'West Bank and Gaza': 'Palestine'
}

# If some entries include multiple countries separated by semicolons, split them
df = df.drop('Country', axis=1).join(
    df['Country'].str.split('; ', expand=True).stack().reset_index(level=1, drop=True).rename('Country')
)

# Apply standardized country names
df['Country'] = df['Country'].replace(country_renames)

# Load open-source geographic data
# 'naturalearth_lowres' is a publicly available dataset integrated into GeoPandas
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Merge geographic data with clustering results
world = world.merge(df, how='left', left_on='name', right_on='Country')

# Define color mapping for clusters
cluster_colors = {0: 'red', 1: 'green', 2: 'blue', 3: 'yellow', 4: 'purple'}

# Create the plot
fig, ax = plt.subplots(1, 1, figsize=(15, 10))

# Plot countries, coloring them based on assigned clusters
world.plot(
    ax=ax,
    color=world['cluster'].map(cluster_colors).fillna('lightgrey'),  # Countries without stock exchanges are in grey
    legend=True
)

# Create a manual legend mapping cluster numbers to colors
legend_labels = {value: f'Cluster {key}' for key, value in cluster_colors.items()}
patches = [
    plt.Line2D([0], [0], marker='o', color=color, label=label, markersize=10, linestyle='None')
    for color, label in legend_labels.items()
]
plt.legend(handles=patches, title='Clusters', loc='center left')

# Remove axis borders and labels for cleaner visualization
ax.set_axis_off()

# Display the map
plt.show()
